In [32]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/alekomamukashvili/my-custom-transformer/my-preprocessing-classes.ipynb
/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow dagshub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 74.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
import sys


sys.path.append('/kaggle/usr/lib/my_preprocessing_classes')
import my_preprocessing_classes as mpc

train_path = '/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv'
train_df = pd.read_csv(train_path, nrows=50000) 


X = train_df.drop('isFraud', axis=1)
y = train_df['isFraud']


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"მონაცემები ჩატვირთულია!")
print(f"სვეტების რაოდენობა: {len(X.columns)}")
print(f"Target (isFraud) ნაპოვნია და გამოყოფილია!")

 მონაცემები ჩატვირთულია!
სვეტების რაოდენობა: 393
 Target (isFraud) ნაპოვნია და გამოყოფილია!


In [9]:
mlflow.set_experiment("Logistic_Regression_Experiment")

pipeline = Pipeline([
    ('cleaner', mpc.FraudDataCleaner(drop_threshold=0.9)),
    ('engineer', mpc.FraudFeatureEngineer()),
    ('encoder', mpc.FraudEncoder()),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear'))
])

with mlflow.start_run(run_name="Logistic_Regression_Base_Run"):
    pipeline.fit(X_train, y_train)
    
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_probs)
    
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_metric("auc", auc_score)
    mlflow.sklearn.log_model(pipeline, "fraud_pipeline_model")
    
    print(f"📊 Validation AUC: {auc_score:.4f}")

2026/05/06 15:51:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/06 15:51:20 INFO mlflow.store.db.utils: Updating database tables
2026/05/06 15:51:24 INFO mlflow.tracking.fluent: Experiment with name 'Logistic_Regression_Experiment' does not exist. Creating a new experiment.
2026/05/06 15:52:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:52:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


📊 Validation AUC: 0.8191


In [11]:
import dagshub
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score

dagshub.init(repo_owner='aleko-mamukashvili', repo_name='-IEEE-CIS-Fraud-Detection.', mlflow=True)

mlflow.set_experiment("Logistic_Regression_Experiment")

with mlflow.start_run(run_name="Final_Production_Run"):
    pipeline.fit(X_train, y_train)
    
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_probs)
    
    mlflow.log_metric("auc", auc_score)
    mlflow.sklearn.log_model(pipeline, "model")
    
    print(f"წარმატებით აიტვირთა aleko-mamukashvili-ს რეპოზიტორიაში!")
    print(f"AUC შედეგი: {auc_score:.4f}")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=769760f0-6d5c-45bd-be95-f10fa550e16e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d612b85534289da1293150fb9fa19b74452fbf79e34a2ea0650517e723a0adcb




Accessing as aleko-mamukashvili

Initialized MLflow to track repo "aleko-mamukashvili/-IEEE-CIS-Fraud-Detection."

Repository aleko-mamukashvili/-IEEE-CIS-Fraud-Detection. initialized!

2026/05/06 16:03:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:03:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ წარმატებით აიტვირთა aleko-mamukashvili-ს რეპოზიტორიაში!
📊 AUC შედეგი: 0.8191
🏃 View run Final_Production_Run at: https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow/#/experiments/0/runs/e4a67f6c956f4d03933f79784c6ed2b2
🧪 View experiment at: https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow/#/experiments/0


In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score

def advanced_engineering(df):
    df = df.copy()
    
    df['hour'] = (df['TransactionDT'] // 3600) % 24
    df['day'] = (df['TransactionDT'] // (3600 * 24)) % 7
    
    df['Amt_to_mean_card1'] = df['TransactionAmt'] / df.groupby('card1')['TransactionAmt'].transform('mean')
    df['Amt_to_std_card1'] = df['TransactionAmt'] / df.groupby('card1')['TransactionAmt'].transform('std')
    
    df['card1_addr1'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str)
    
    df['P_emaildomain'] = df['P_emaildomain'].fillna('missing')
    df['P_email_suffix'] = df['P_emaildomain'].map(lambda x: str(x).split('.')[-1])
    
    return df

X_train_eng = advanced_engineering(X_train)
X_val_eng = advanced_engineering(X_val)

with mlflow.start_run(run_name="Engineering_Run_Clean"):
    pipeline.fit(X_train_eng, y_train)
    
    probs = pipeline.predict_proba(X_val_eng)[:, 1]
    auc_score = roc_auc_score(y_val, probs)
    
    mlflow.log_metric("auc", auc_score)
    mlflow.log_param("features_count", X_train_eng.shape[1])
    mlflow.sklearn.log_model(pipeline, "model")
    
    print(f"📊 AUC: {auc_score:.4f}")

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

def advanced_engineering(df):
    df = df.copy()

    df['hour'] = (df['TransactionDT'] // 3600) % 24
    df['day'] = (df['TransactionDT'] // (3600 * 24)) % 7

    df['Amt_to_mean_card1'] = df['TransactionAmt'] / df.groupby('card1')['TransactionAmt'].transform('mean')
    df['Amt_to_std_card1'] = df['TransactionAmt'] / df.groupby('card1')['TransactionAmt'].transform('std')
    

    df.replace([np.inf, -np.inf], 0, inplace=True)

    df.fillna(0, inplace=True)
    
    df['card1_addr1'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str)
    
    df['P_emaildomain'] = df['P_emaildomain'].fillna('missing')
    df['P_email_suffix'] = df['P_emaildomain'].map(lambda x: str(x).split('.')[-1])
    
    return df

X_train_eng = advanced_engineering(X_train)
X_val_eng = advanced_engineering(X_val)

pipeline.fit(X_train_eng, y_train)

probs = pipeline.predict_proba(X_val_eng)[:, 1]
auc_score = roc_auc_score(y_val, probs)

print(f"📊 Local AUC: {auc_score:.4f}")

📊 Local AUC: 0.8473


In [16]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='aleko-mamukashvili', repo_name='-IEEE-CIS-Fraud-Detection.', mlflow=True)
mlflow.set_experiment("Logistic_Regression_Experiment")

with mlflow.start_run(run_name="Feature_Engineering_Success"):
    pipeline.fit(X_train_eng, y_train)
    
    probs = pipeline.predict_proba(X_val_eng)[:, 1]
    auc_score = roc_auc_score(y_val, probs)
    
    mlflow.log_metric("auc", auc_score)
    mlflow.log_param("engineered", True)
    mlflow.log_param("feature_count", X_train_eng.shape[1])
    mlflow.sklearn.log_model(pipeline, "engineered_model_v1")
    
    print(f"✅ აიტვირთა DagsHub-ზე! AUC: {auc_score:.4f}")

Initialized MLflow to track repo "aleko-mamukashvili/-IEEE-CIS-Fraud-Detection."

Repository aleko-mamukashvili/-IEEE-CIS-Fraud-Detection. initialized!

2026/05/06 16:13:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:13:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ აიტვირთა DagsHub-ზე! AUC: 0.8473
🏃 View run Feature_Engineering_Success at: https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow/#/experiments/0/runs/7768a625094d4898bc47e2c496ba3da1
🧪 View experiment at: https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow/#/experiments/0


In [17]:
import dagshub
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score


dagshub.init(repo_owner='aleko-mamukashvili', repo_name='-IEEE-CIS-Fraud-Detection.', mlflow=True)
mlflow.set_experiment("Logistic_Regression_Experiment")


train_probs = pipeline.predict_proba(X_train_eng)[:, 1]
train_auc = roc_auc_score(y_train, train_probs)

val_probs = pipeline.predict_proba(X_val_eng)[:, 1]
val_auc = roc_auc_score(y_val, val_probs)

gap = train_auc - val_auc


with mlflow.start_run(run_name="Bias_Variance_Test"):
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("auc_gap", gap)
    

    if gap > 0.05:
        mlflow.set_tag("issue", "High Variance / Overfitting")
    elif val_auc < 0.80:
        mlflow.set_tag("issue", "High Bias / Underfitting")
    else:
        mlflow.set_tag("issue", "Balanced")

    mlflow.sklearn.log_model(pipeline, "model_bias_test")
    
    print(f" Train AUC: {train_auc:.4f}")
    print(f" Validation AUC: {val_auc:.4f}")
    print(f" Gap: {gap:.4f}")
    print(f" შედეგები აიტვირთა DagsHub-ზე!")

Initialized MLflow to track repo "aleko-mamukashvili/-IEEE-CIS-Fraud-Detection."

Repository aleko-mamukashvili/-IEEE-CIS-Fraud-Detection. initialized!

2026/05/06 16:15:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:15:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


 Train AUC: 0.8921
 Validation AUC: 0.8473
 Gap: 0.0447
 შედეგები აიტვირთა DagsHub-ზე!
🏃 View run Bias_Variance_Test at: https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow/#/experiments/0/runs/b2a27a1684a646ee8beeca67014ee3d4
🧪 View experiment at: https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow/#/experiments/0
